    # What drives demand and price: statistical notes

    Quick statistical pass over two years of hourly system data (consumption, temperature, day-ahead price)
    to answer three questions for the trading desk: how big is the weekend effect, does price drive
    demand, and how well does a simple linear model explain daily demand.


> **Fixed version.** Each correction is marked with a *Fix:* note.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df["consumption_mwh"].to_numpy()
price = df["price_eur_mwh"].to_numpy()
temp = df["temp_c"].to_numpy()
hours = df["time"].dt.hour.to_numpy()
dow = df["time"].dt.dayofweek.to_numpy()
year = df["time"].dt.year.to_numpy()
print(df.shape, df["time"].min(), df["time"].max())

(17520, 6) 2022-01-01 00:00:00+00:00 2023-12-31 23:00:00+00:00


## Sanity checks

NumPy and pandas summary statistics should agree exactly.

*Fix:* the std rows differ because `np.std` uses `ddof=0` (population) and pandas uses `ddof=1` (sample). It is not rounding: the ratio is exactly $\sqrt{n/(n-1)}$. Pass `ddof=1` to NumPy (or `ddof=0` to pandas) when comparing.

In [2]:
n = len(cons)
checks = pd.DataFrame({
    "numpy ddof=0": [np.std(cons), np.std(price)],
    "numpy ddof=1": [np.std(cons, ddof=1), np.std(price, ddof=1)],
    "pandas":       [df["consumption_mwh"].std(), df["price_eur_mwh"].std()],
}, index=["cons std", "price std"])
checks["ratio pandas/np0"] = checks["pandas"] / checks["numpy ddof=0"]
print("sqrt(n/(n-1)) =", np.sqrt(n / (n - 1)))
checks

sqrt(n/(n-1)) = 1.0000285400345392


,numpy ddof=0,numpy ddof=1,pandas,ratio pandas/np0
cons std,4204.818865,4204.938871,4204.938871,1.000029
price std,36.897101,36.898154,36.898154,1.000029


## Temperature distribution

Readings below -5C are treated as sensor errors and removed.

*Fix:* (a) -6C in a British winter is not a sensor error, so the mask is a research choice that deletes exactly the cold hours that drive heating demand; (b) once NaN is present, `np.mean` / `np.percentile` return `nan`. Use `np.nanmean` / `np.nanpercentile` and report how many values were masked.

In [3]:
mask = temp < -5
print(f"{mask.sum()} hours below -5C ({mask.mean():.2%}); min temp {temp.min():.1f}C")
temp_clean = temp.copy()
temp_clean[mask] = np.nan
temp_stats = pd.DataFrame({
    "all data":   [len(temp), np.mean(temp), *np.percentile(temp, [5, 50, 95])],
    "masked <-5": [np.sum(~mask), np.nanmean(temp_clean), *np.nanpercentile(temp_clean, [5, 50, 95])],
}, index=["n", "mean", "p05", "p50", "p95"])
temp_stats.round(2)

41 hours below -5C (0.23%); min temp -6.4C


,all data,masked <-5
n,17520.00,17479.00
mean,9.93,9.97
p05,-0.59,-0.50
p50,9.96,9.98
p95,20.49,20.50


## Daily profile

Centre the midnight values as a quick check that the overnight level is stable, then compute the average hour-of-day profile.

*Fix:* `cons[::24]` is a **view**, so `midnight -= midnight.mean()` overwrote every midnight value in `cons` itself with a deviation (hour 0 of the original profile is about 0 instead of 23,000). Use `.copy()` before modifying a slice. Also, dividing the mean profile by its sum does not give a "probability of peak": to estimate that, count on which hour each day's maximum falls.

In [4]:
midnight = cons[::24].copy()
midnight -= midnight.mean()
print("midnight deviations: mean %.2f, sd %.1f" % (midnight.mean(), midnight.std()))
print("cons still intact at hour 0:", cons[0], cons[24])

midnight deviations: mean -0.00, sd 2542.1
cons still intact at hour 0: 26858.4 26785.3


In [5]:
profile = pd.Series(cons).groupby(hours).mean()
daily_peak_hour = pd.Series(cons).groupby(df["time"].dt.floor("D").to_numpy()).idxmax().map(lambda i: hours[i])
p_peak = daily_peak_hour.value_counts(normalize=True).reindex(range(24), fill_value=0)
pd.DataFrame({"mean_mwh": profile.round(0), "share_of_mean": (profile / profile.sum()).round(3),
              "p_daily_peak": p_peak.round(3)}).T

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
mean_mwh,25430.000,24372.000,23875.000,23610.000,23883.000,24989.000,27230.000,30060.000,31620.000,32025.000,...,29786.000,30079.000,31896.000,34343.000,35363.00,34008.000,32134.000,30268.000,28044.00,26420.000
share_of_mean,0.036,0.035,0.034,0.034,0.034,0.036,0.039,0.043,0.045,0.046,...,0.042,0.043,0.045,0.049,0.05,0.048,0.046,0.043,0.04,0.038
p_daily_peak,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.001,0.001,...,0.000,0.000,0.000,0.033,0.96,0.004,0.000,0.000,0.00,0.000


## Annual energy

*Fix:* `astype(int) // 1000` floors every hour to whole GWh, throwing away up to 999 MWh per row (about 0.5 GWh on average, times 8,760 hours a year). Convert with float division and sum.

In [6]:
annual_wrong = (df["consumption_mwh"].astype(int) // 1000).groupby(year).sum()
annual = (df["consumption_mwh"] / 1000).groupby(year).sum()
pd.DataFrame({"floored GWh": annual_wrong, "GWh": annual.round(0), "lost GWh": (annual - annual_wrong).round(0)})

,floored GWh,GWh,lost GWh
2022,253230,257609.0,4379.0
2023,251615,255995.0,4380.0


## Weekend effect

Ops claim weekends are about 2,000 MWh/h lower than weekdays.

*Fix:* `dayofweek >= 6` is Sunday only (Monday = 0, Sunday = 6). Saturday was counted as a weekday, diluting both groups. The t-test on 17k hourly observations also treats every hour as independent; with lag-1 autocorrelation above 0.9 the effective sample size is far smaller, so the t-statistic is inflated by roughly $\sqrt{(1+\rho)/(1-\rho)}$. Test on daily means (still autocorrelated, but much less) or use a block bootstrap.

In [7]:
weekend = cons[dow >= 5]
weekday = cons[dow < 5]
effect = weekend.mean() - weekday.mean()
rho = np.corrcoef(cons[1:], cons[:-1])[0, 1]
t_hourly, p_hourly = stats.ttest_ind(weekend, weekday)
daily = df.set_index("time")["consumption_mwh"].resample("D").mean()
is_we = daily.index.dayofweek >= 5
t_daily, p_daily = stats.ttest_ind(daily[is_we], daily[~is_we])
print(f"Sunday-only definition : {cons[dow >= 6].mean() - cons[dow < 6].mean():,.0f} MWh/h")
print(f"Sat+Sun definition     : {effect:,.0f} MWh/h")
print(f"lag-1 autocorr {rho:.3f} -> naive t inflated by ~x{np.sqrt((1 + rho) / (1 - rho)):.1f}")
print(f"hourly t-test: t = {t_hourly:.1f}, p = {p_hourly:.1e}; daily-mean t-test: t = {t_daily:.1f}, p = {p_daily:.1e}")

Sunday-only definition : -1,750 MWh/h
Sat+Sun definition     : -2,082 MWh/h
lag-1 autocorr 0.935 -> naive t inflated by ~x5.5
hourly t-test: t = -30.4, p = 1.8e-198; daily-mean t-test: t = -12.7, p = 2.7e-33


## Price and demand

*Fix:* both series share the hour-of-day cycle, the heating season and the 2022 gas regime, so the level correlation is mostly common seasonality, not a price effect. Remove the shared structure (demean by hour, or difference) before correlating; the p-value from `pearsonr` assumes independent observations and is meaningless on 17k autocorrelated hours. `np.corrcoef(price[1:], cons[:-1])` pairs price at *t+1* with consumption at *t*, so it measures whether consumption leads price, the opposite of the text. Also, price is set day-ahead from forecast demand, so the causal story runs demand -> price.

In [8]:
r_level = np.corrcoef(price, cons)[0, 1]
cons_dm = cons - pd.Series(cons).groupby(hours).transform("mean").to_numpy()
price_dm = price - pd.Series(price).groupby(hours).transform("mean").to_numpy()
r_dm = np.corrcoef(price_dm, cons_dm)[0, 1]
r_diff = np.corrcoef(np.diff(price), np.diff(cons))[0, 1]
n_eff = len(cons) * (1 - rho) / (1 + rho)
print(f"level corr          : {r_level:.3f}")
print(f"hour-demeaned corr  : {r_dm:.3f}")
print(f"first-difference corr: {r_diff:.3f}")
print(f"effective n for the level test ~ {n_eff:,.0f} instead of {len(cons):,}")
print(f"corr(price_t+1, cons_t) = {np.corrcoef(price[1:], cons[:-1])[0, 1]:.3f}   <- consumption leads price (what the original computed)")
print(f"corr(price_t-1, cons_t) = {np.corrcoef(price[:-1], cons[1:])[0, 1]:.3f}   <- price leads consumption (what the text claimed)")

level corr          : 0.576
hour-demeaned corr  : 0.389
first-difference corr: 0.373
effective n for the level test ~ 588 instead of 17,520
corr(price_t+1, cons_t) = 0.544   <- consumption leads price (what the original computed)
corr(price_t-1, cons_t) = 0.521   <- price leads consumption (what the text claimed)


## Linear model of daily demand

OLS of daily mean consumption on daily mean temperature and price, solved with the normal equations.

*Fix:* three things. (1) No intercept column: the regression is forced through the origin, so the coefficients absorb the mean. (2) `ss_tot = (y**2).sum()` is uncentred; with a mean of ~29,000 it is enormous, which is why $R^2$ looked like 0.97. (3) `beta` was solved as a `(2,1)` column, so `pred` is `(n,1)` and `y - pred` broadcasts to an `(n,n)` matrix of every $y_i - \hat y_j$ pair: its mean is ~0 and its sd is huge. Keep everything 1-D and check `.shape`.

In [9]:
daily = df.set_index("time").resample("D").mean(numeric_only=True)
y = daily["consumption_mwh"].to_numpy()
X = np.column_stack([np.ones(len(daily)), daily["temp_c"].to_numpy(), daily["price_eur_mwh"].to_numpy()])
beta = np.linalg.solve(X.T @ X, X.T @ y)          # 1-D, shape (3,)
pred = X @ beta                                     # shape (n,)
resid = y - pred
print("shapes:", y.shape, pred.shape, resid.shape)
ss_res = (resid ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
print("beta [const, temp, price]:", beta.round(3))
print(f"centred R2 = {1 - ss_res / ss_tot:.3f}   (uncentred 'R2' would be {1 - ss_res / (y ** 2).sum():.3f})")
print(f"residual mean {resid.mean():.1f}, sd {resid.std():.0f}, RMSE {np.sqrt((resid ** 2).mean()):.0f} MWh/h")

shapes: (730,) (730,) (730,)
beta [const, temp, price]: [30715.034  -284.018    14.421]
centred R2 = 0.761   (uncentred 'R2' would be 0.999)
residual mean 0.0, sd 1086, RMSE 1086 MWh/h


*Fix:* `==` on floats from two different solvers fails on the last bit. Use `np.allclose`.

In [10]:
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("exactly equal:", np.mean(beta == beta_lstsq), "  allclose:", np.allclose(beta, beta_lstsq))
print("max abs diff:", np.abs(beta - beta_lstsq).max())

exactly equal: 0.0   allclose: True
max abs diff: 1.1168594937771559e-09


## Bootstrap confidence interval for the price-demand correlation

*Fix:* resampling individual hours i.i.d. destroys the autocorrelation, so the interval is far too narrow; resample blocks (here: whole weeks) instead. Seed the generator so the number is reproducible.

In [11]:
rng = np.random.default_rng(0)
n = len(cons)
block = 24 * 7
starts_all = np.arange(0, n - block + 1)
n_blocks = n // block
boot_iid, boot_block = [], []
for _ in range(500):
    idx = rng.integers(0, n, n)
    boot_iid.append(np.corrcoef(price[idx], cons[idx])[0, 1])
    starts = rng.choice(starts_all, n_blocks)
    idx = (starts[:, None] + np.arange(block)).ravel()
    boot_block.append(np.corrcoef(price_dm[idx], cons_dm[idx])[0, 1])
print("iid bootstrap, level corr       : [%.3f, %.3f]" % tuple(np.percentile(boot_iid, [2.5, 97.5])))
print("weekly-block bootstrap, demeaned: [%.3f, %.3f]" % tuple(np.percentile(boot_block, [2.5, 97.5])))

iid bootstrap, level corr       : [0.565, 0.587]
weekly-block bootstrap, demeaned: [0.313, 0.468]


## Results

In [12]:
summary = pd.Series({
    "annual GWh 2022": round(annual.loc[2022]),
    "annual GWh 2023": round(annual.loc[2023]),
    "weekend effect MWh/h (Sat+Sun)": round(effect),
    "temp p95 (C)": round(np.percentile(temp, 95), 2),
    "corr(price, cons) level": round(r_level, 3),
    "corr(price, cons) hour-demeaned": round(r_dm, 3),
    "corr(price, cons) differenced": round(r_diff, 3),
    "block-bootstrap CI (demeaned)": "[%.3f, %.3f]" % tuple(np.percentile(boot_block, [2.5, 97.5])),
    "daily model R2 (centred)": round(1 - ss_res / ss_tot, 3),
    "daily model RMSE": round(np.sqrt((resid ** 2).mean())),
    "most likely daily peak hour": int(p_peak.idxmax()),
})
print(summary.to_string())
print()
print("Conclusions: the weekend effect is ~2,100 MWh/h, in line with ops; price and demand co-move because")
print("both follow the daily/seasonal cycle and demand feeds into the day-ahead price, not the other way round;")
print("temperature + price explain a moderate share of daily demand variance, not 97%.")

annual GWh 2022                            257609
annual GWh 2023                            255995
weekend effect MWh/h (Sat+Sun)              -2082
temp p95 (C)                                20.49
corr(price, cons) level                     0.576
corr(price, cons) hour-demeaned             0.389
corr(price, cons) differenced               0.373
block-bootstrap CI (demeaned)      [0.313, 0.468]
daily model R2 (centred)                    0.761
daily model RMSE                             1086
most likely daily peak hour                    18

Conclusions: the weekend effect is ~2,100 MWh/h, in line with ops; price and demand co-move because
both follow the daily/seasonal cycle and demand feeds into the day-ahead price, not the other way round;
temperature + price explain a moderate share of daily demand variance, not 97%.
